## Demo Run

This demo notebook was run in Google Colab with T4 GPU. It clones the repository, creates a folder in which to run, and installs requirements. Then it first trains the base VAE model, runs evaluation metrics on it and runs TSTR classifier on it. Then, it finetunes the trained VAE using the GAN, then runs the evaluation metrics, and finally runs the TSTR classifier on it. Outputs of each step are saved to subfolders within the created demo folder. As a note, the hyperparameters that are used in this notebook are just for demonstration purposes and don't necessarily reflect the final model chosen for the project.

In [1]:
!git clone https://github.com/Nagham-Sabbour/Synthetic-ECG-Generator

Cloning into 'Synthetic-ECG-Generator'...
remote: Enumerating objects: 772, done.
remote: Counting objects: 100% (695/695), done.
remote: Compressing objects: 100% (580/580), done.
remote: Total 772 (delta 184), reused 597 (delta 115), pack-reused 77 (from 3)
Receiving objects: 100% (772/772), 770.02 MiB | 36.32 MiB/s, done.
Resolving deltas: 100% (216/216), done.
Updating files: 100% (481/481), done.


In [2]:
%cd Synthetic-ECG-Generator
!mkdir demo
%cd demo

/content/Synthetic-ECG-Generator
/content/Synthetic-ECG-Generator/demo


In [10]:
!pip install -r ../requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 19.4 MB/s eta 0:00:00


Train the VAE (outputs will be saved to /demo/checkpoints, /demo/training_plots, /demo/visuals):

In [6]:
!python ../train.py --epochs 50 --lr 1e-3 --embedding-dim 64 --loss-beta 0.5 --data-root ../processed_data

Epoch [1/50]  Total Loss: 21835.433  Reconstruction Loss: 19183.376  KL Divergence Loss: 5304.114
Epoch [2/50]  Total Loss: 10997.731  Reconstruction Loss: 7447.513  KL Divergence Loss: 7100.435
Epoch [3/50]  Total Loss: 9190.528  Reconstruction Loss: 5581.820  KL Divergence Loss: 7217.414
Epoch [4/50]  Total Loss: 8456.199  Reconstruction Loss: 4834.145  KL Divergence Loss: 7244.108
Epoch [5/50]  Total Loss: 7895.854  Reconstruction Loss: 4277.164  KL Divergence Loss: 7237.380
Epoch [6/50]  Total Loss: 7672.779  Reconstruction Loss: 4046.297  KL Divergence Loss: 7252.964
Epoch [7/50]  Total Loss: 7450.380  Reconstruction Loss: 3839.437  KL Divergence Loss: 7221.887
Epoch [8/50]  Total Loss: 7211.033  Reconstruction Loss: 3598.393  KL Divergence Loss: 7225.281
Epoch [9/50]  Total Loss: 7135.748  Reconstruction Loss: 3525.458  KL Divergence Loss: 7220.579
Epoch [10/50]  Total Loss: 6973.500  Reconstruction Loss: 3375.641  KL Divergence Loss: 7195.719
Epoch [11/50]  Total Loss: 6899.645 

Run evaluation on the trained VAE before moving on to finetuning (outputs will be saved in /demo/test_runs):

In [13]:
!python ../evaluate.py --checkpoint-path checkpoints/vae_best_20260814_231245.pt  --embedding-dim 64 --data-root ../processed_data

Loaded VAE checkpoint from checkpoints/vae_best_20260814_231245.pt (epoch=49, validation_loss=5852.686491243132)
Saved generated samples plot to test_runs/vae_best_20260814_231245/generated_samples_20260814_234711.png

PSD scores:
    class_id class_name  log_psd_mse
0          0         CD     1.432418
1          1     CD+HYP     1.551521
2          2      CD+MI     1.466961
3          3    CD+NORM     1.512534
4          4    CD+STTC     1.351607
5          5        HYP     1.641193
6          6   HYP+STTC     1.363238
7          7         MI     1.349844
8          8    MI+STTC     1.403706
9          9       NORM     1.500226
10        10       STTC     1.303292

R-peak plausibility summary:
       source class_name  samples  ...  mean_rr_ms  mean_hr_bpm  rhythm_match_rate
0   Generated         CD      100  ...  683.465079    90.895783              95.00
1   Generated     CD+HYP       54  ...  632.800485    96.941413              64.81
2   Generated      CD+MI      100  ...  703.20

Run TSTR on the trained VAE before moving on to finetuning (outputs will be saved in /demo/test_runs):

In [14]:
!python ../tstr.py --checkpoint-path checkpoints/vae_best_20260814_231245.pt  --embedding-dim 64 --classifier-epochs 50 --classifier-patience 12 --learning-rate 1e-3 --data-root ../processed_data

Loaded VAE checkpoint from checkpoints/vae_best_20260814_231245.pt (epoch=49, validation_loss=5852.686491243132)

Training classifier on real ECGs...
Classifier epoch 1/50 | train loss: 2.1950 | validation macro F1: 0.2009
Classifier epoch 2/50 | train loss: 1.9993 | validation macro F1: 0.2704
Classifier epoch 3/50 | train loss: 1.9255 | validation macro F1: 0.2935
Classifier epoch 4/50 | train loss: 1.8715 | validation macro F1: 0.2884
Classifier epoch 5/50 | train loss: 1.8460 | validation macro F1: 0.3043
Classifier epoch 6/50 | train loss: 1.8213 | validation macro F1: 0.2086
Classifier epoch 7/50 | train loss: 1.8109 | validation macro F1: 0.2618
Classifier epoch 8/50 | train loss: 1.7928 | validation macro F1: 0.3017
Classifier epoch 9/50 | train loss: 1.7834 | validation macro F1: 0.2764
Classifier epoch 10/50 | train loss: 1.7815 | validation macro F1: 0.2831
Classifier epoch 11/50 | train loss: 1.7827 | validation macro F1: 0.2751
Classifier epoch 12/50 | train loss: 1.7489 |

Finetune the trained VAE using GAN (outputs will be saved to /demo/checkpoints, /demo/training_plots, /demo/visuals):

In [7]:
!python ../finetune.py --trained-vae-filename vae_best_20260814_231245.pt --epochs 50 --decoder-lr 1e-3 --discrim-lr 1e-4 --embedding-dim 64 --loss-lambda-adv 50 --data-root ../processed_data

Loaded VAE checkpoint from checkpoints/vae_best_20260814_231245.pt (epoch=49, validation_loss=5852.686491243132)
Epoch [1/50]  Generator Loss: 2337.191  Reconstruction Loss: 2302.008  Generator Adversarial Loss: 0.704  Discriminator Loss: 1.375
Epoch [2/50]  Generator Loss: 2319.416  Reconstruction Loss: 2278.371  Generator Adversarial Loss: 0.821  Discriminator Loss: 1.263
Epoch [3/50]  Generator Loss: 2344.874  Reconstruction Loss: 2293.396  Generator Adversarial Loss: 1.030  Discriminator Loss: 1.071
Epoch [4/50]  Generator Loss: 2375.632  Reconstruction Loss: 2315.790  Generator Adversarial Loss: 1.197  Discriminator Loss: 0.958
Epoch [5/50]  Generator Loss: 2322.414  Reconstruction Loss: 2260.570  Generator Adversarial Loss: 1.237  Discriminator Loss: 0.949
Epoch [6/50]  Generator Loss: 2339.689  Reconstruction Loss: 2280.493  Generator Adversarial Loss: 1.184  Discriminator Loss: 0.992
Epoch [7/50]  Generator Loss: 2350.830  Reconstruction Loss: 2292.789  Generator Adversarial Lo

Run evaluation on finetuned model (outputs will be saved in /demo/test_runs):

In [11]:
!python ../evaluate.py --checkpoint-path checkpoints/vae_gan_best_20260814_231729.pt  --embedding-dim 64 --data-root ../processed_data

Loaded VAE checkpoint from checkpoints/vae_gan_best_20260814_231729.pt (epoch=1, validation_loss=2337.191130250365)
Saved generated samples plot to test_runs/vae_gan_best_20260814_231729/generated_samples_20260814_232706.png

PSD scores:
    class_id class_name  log_psd_mse
0          0         CD     1.474112
1          1     CD+HYP     1.604565
2          2      CD+MI     1.522313
3          3    CD+NORM     1.579448
4          4    CD+STTC     1.407798
5          5        HYP     1.698783
6          6   HYP+STTC     1.425792
7          7         MI     1.410952
8          8    MI+STTC     1.481670
9          9       NORM     1.558676
10        10       STTC     1.366155

R-peak plausibility summary:
       source class_name  samples  ...  mean_rr_ms  mean_hr_bpm  rhythm_match_rate
0   Generated         CD      100  ...  701.447857    88.757131              97.00
1   Generated     CD+HYP       54  ...  655.241843    93.580727              74.07
2   Generated      CD+MI      100  ... 

Run TSTR on the finetuned model (outputs will be saved in /demo/test_runs):

In [12]:
!python ../tstr.py --checkpoint-path checkpoints/vae_gan_best_20260814_231729.pt  --embedding-dim 64 --classifier-epochs 50 --classifier-patience 12 --learning-rate 1e-3 --data-root ../processed_data

Loaded VAE checkpoint from checkpoints/vae_gan_best_20260814_231729.pt (epoch=1, validation_loss=2337.191130250365)

Training classifier on real ECGs...
Classifier epoch 1/50 | train loss: 2.1950 | validation macro F1: 0.2012
Classifier epoch 2/50 | train loss: 1.9993 | validation macro F1: 0.2711
Classifier epoch 3/50 | train loss: 1.9256 | validation macro F1: 0.2945
Classifier epoch 4/50 | train loss: 1.8716 | validation macro F1: 0.2888
Classifier epoch 5/50 | train loss: 1.8464 | validation macro F1: 0.3008
Classifier epoch 6/50 | train loss: 1.8214 | validation macro F1: 0.2100
Classifier epoch 7/50 | train loss: 1.8112 | validation macro F1: 0.2627
Classifier epoch 8/50 | train loss: 1.7928 | validation macro F1: 0.2977
Classifier epoch 9/50 | train loss: 1.7837 | validation macro F1: 0.2782
Classifier epoch 10/50 | train loss: 1.7816 | validation macro F1: 0.2826
Classifier epoch 11/50 | train loss: 1.7826 | validation macro F1: 0.2733
Classifier epoch 12/50 | train loss: 1.748